# V7 — S5 RANSAC: một warp / hypothesis (kế thừa V5, chờ V6)

V7 giữ nguyên toàn bộ pipeline của **V5** (V1 S1–S3, V2 S4 tiling, V3/V4 S6, GPU data residency) và **chỉ đổi cách tính inlier score ở S5**:
- `estimate_hypotheses_v7_kernel`: DLT 4 điểm, 1 thread/hypothesis (như V1).
- `score_hypotheses_warp_v7_kernel`: **1 warp chấm 1 hypothesis**, `__ballot_sync` + `__popc` đếm inlier, tổng lỗi cộng đúng thứ tự match (trùng V5 từng bit).
- `refine_best_warp_v7_kernel`: warp argmax chọn H tốt nhất, warp compaction có thứ tự cho inlier, đếm inlier H refined bằng ballot; DLT least-squares giữ nguyên.

Giữ nguyên dãy mẫu MT19937, 2.000 vòng, threshold 3 px và refinement của V0. V6 chưa ghép nên so sánh hợp lệ là **V5 → V7**.

Chọn Colab **GPU**, rồi **Run all**. Trình tự: build → parity (V0/V5) → sanitizer → baseline V5 xác nhận tỷ trọng S5 → benchmark 12 case → Nsight kernel S5.

In [ ]:
from pathlib import Path
import csv
import datetime
import json
import os
import shlex
import shutil
import subprocess
import sys
from statistics import geometric_mean

REPO_URL = "https://github.com/nganbee/panorama-stitching-project.git"
BRANCH = "v7"  # đổi thành tên nhánh bạn push V7
REPO = Path("/content/panorama-stitching-project")
UPDATE_REPO = True  # False: giữ source đã upload/chỉnh tại Colab.
RUNS = 5
RUN_PROFILER = True
COMMAND_LOG_DIR = None
COMMAND_INDEX = 0

def run(args, check=True, log=None):
    global COMMAND_INDEX
    command = shlex.join([str(a) for a in args])
    print("$", command, flush=True)
    if log is None and COMMAND_LOG_DIR is not None:
        COMMAND_INDEX += 1
        log = COMMAND_LOG_DIR / f"command_{COMMAND_INDEX:03d}.log"
    stream = None
    try:
        if log is not None:
            log = Path(log)
            log.parent.mkdir(parents=True, exist_ok=True)
            stream = log.open("w", encoding="utf-8")
            stream.write("$ " + command + "\n")
        with subprocess.Popen([str(a) for a in args], text=True,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                if stream is not None:
                    stream.write(line)
                    stream.flush()
            code_ret = process.wait()
    finally:
        if stream is not None:
            stream.close()
    if check and code_ret:
        raise RuntimeError(f"Command failed ({code_ret}); log: {log}")
    return code_ret

if not REPO.exists():
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
if UPDATE_REPO:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], text=True).strip()
    if dirty:
        print("Có thay đổi local: giữ nguyên checkout, không tự pull.")
    else:
        run(["git", "fetch", "origin", BRANCH])
        run(["git", "checkout", BRANCH])
        run(["git", "merge", "--ff-only", f"origin/{BRANCH}"])

required = [
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v3_stage6.cu",
    "src/stage_implementations/v4_stage6.cu",
    "src/stage_implementations/v5_stage4.cu",
    "src/stage_implementations/v5_stage6.cu",
    "src/stage_implementations/v7_stage5.cu",
    "src/cuda/v5_pipeline.cu",
    "src/cuda/v7_pipeline.cu",
    "src/tools/v7/test_v7_pipeline.cpp",
]
assert all(Path(p).is_file() for p in required), "Thiếu source V2/V3/V4/V5/V7 tích lũy."
assert RUNS >= 5
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
SESSION_DIR = Path("out") / f"v7_inspect_{STAMP}"
SESSION_DIR.mkdir(parents=True)
COMMAND_LOG_DIR = SESSION_DIR / "commands"
RESULTS = SESSION_DIR / "benchmark"
RESULTS.mkdir(parents=True)
Path("build").mkdir(exist_ok=True)
run(["git", "status", "--short"])
run(["git", "branch", "--show-current"])
run(["git", "rev-parse", "HEAD"])
print("Session:", SESSION_DIR)

## 1. GPU, compiler và build chung (V0–V5, V7)

Giữ `--fmad=false`, không dùng fast math. Một binary `build/stitch_v7` cung cấp V0/V1/V2/V3/V4/V5/V7 với cùng input, flags và GPU session.

In [ ]:
run(["nvidia-smi"], log=SESSION_DIR / "nvidia-smi.log")
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name,uuid,driver_version",
                               "--format=csv,noheader"], text=True).strip()
print("GPU:", GPU)
run(["nvcc", "--version"])
FLAGS = [
    "nvcc", "-O2", "-std=c++17", "--fmad=false",
    "-DPANORAMA_WITH_CUDA", "-DPANORAMA_WITH_V2", "-DPANORAMA_WITH_V3",
    "-DPANORAMA_WITH_V4", "-DPANORAMA_WITH_V5", "-DPANORAMA_WITH_V7",
    "-Isrc/core_headers", "-Isrc/libraries",
]
SOURCES = [
    "src/integration/v0_pipeline.cpp",
    "src/stage_implementations/v0_stage12.cpp",
    "src/stage_implementations/v0_stage34.cpp",
    "src/stage_implementations/v0_stage5.cpp",
    "src/stage_implementations/v0_stage6.cpp",
    "src/stage_implementations/v1_stage12.cu",
    "src/stage_implementations/v1_stage34.cu",
    "src/stage_implementations/v1_stage5.cu",
    "src/stage_implementations/v1_stage6.cu",
    "src/stage_implementations/v2_stage4.cu",
    "src/stage_implementations/v3_stage6.cu",
    "src/stage_implementations/v4_stage6.cu",
    "src/stage_implementations/v5_stage4.cu",
    "src/stage_implementations/v5_stage6.cu",
    "src/stage_implementations/v7_stage5.cu",
    "src/cuda/cuda_pipeline.cu",
    "src/cuda/v2_pipeline.cu",
    "src/cuda/v3_pipeline.cu",
    "src/cuda/v4_pipeline.cu",
    "src/cuda/v5_pipeline.cu",
    "src/cuda/v7_pipeline.cu",
    "src/libraries/stb_impl.cpp",
]
run(FLAGS + ["src/integration/main.cpp"] + SOURCES + ["-o", "build/stitch_v7"])
run(["build/stitch_v7", "--list-versions"])
run(["build/stitch_v7", "--dump-params"])
(SESSION_DIR / "environment.json").write_text(json.dumps({
    "gpu": GPU,
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
    "flags": FLAGS,
    "runs": RUNS,
    "warmup_runs": 1,
}, indent=2), encoding="utf-8")

## 2. Parity V7 vs V0/V5

`test_v7_pipeline` kiểm tra:
- **S5 riêng**: `v7::ransac_homography` trùng `v5::ransac_homography` từng bit (H, inliers, status) với n = 4…3000 (quanh bội số 32), outlier, điểm trùng (tie-break), match không hợp lệ; và khớp V0 (cùng số inlier, sai số chiếu ≤ 1e-3 px). In thời gian S5 V5 vs V7 ở n = 3000.
- **Pipeline**: V7 trùng V5 từng bit (keypoints, matches, H, panorama, số byte copy) và đạt tiêu chí PLAN so với V0.

In [ ]:
run(FLAGS + ["src/tools/v7/test_v7_pipeline.cpp"] + SOURCES + ["-o", "build/test_v7_pipeline"])
run(["build/test_v7_pipeline"])
run(["build/stitch_v7", "--selftest", "--version", "7"])

## 3. Compute Sanitizer (`memcheck`, `racecheck`)

In [ ]:
if shutil.which("compute-sanitizer"):
    run(["compute-sanitizer", "--tool", "memcheck", "--error-exitcode", "1", "build/test_v7_pipeline"])
    run(["compute-sanitizer", "--tool", "racecheck", "--error-exitcode", "1",
         "build/stitch_v7", "--selftest", "--version", "7"])
else:
    print("NOT RUN: Compute Sanitizer unavailable.")

## 4. Baseline V5 (thay cho V6 chưa ghép): xác nhận tỷ trọng S5 trước khi đo V7

PLAN: chỉ chốt V7 khi timing version liền trước cho thấy S5 đáng tối ưu.

In [ ]:
BASELINE_ROWS = []
for scene, folder in [("location1", "small_720p"), ("location1", "large_4k")]:
    left = f"dataset/{scene}/{folder}/1.jpg"
    right = f"dataset/{scene}/{folder}/2.jpg"
    report = SESSION_DIR / f"v5_baseline_{folder}.json"
    run(["build/stitch_v7", left, right, SESSION_DIR / f"v5_baseline_{folder}.png",
         "--version", "5", "--runs", RUNS, "--json", report], log=SESSION_DIR / f"v5_baseline_{folder}.log")
    t = json.loads(report.read_text(encoding="utf-8"))["timings_ms"]
    stages = ["gray", "blur", "harris", "nms_topk", "describe", "match", "ransac", "warp_blend"]
    share = {s: 100 * t[s] / t["kernel_total"] for s in stages}
    BASELINE_ROWS.append({"case": f"{scene}_{folder}", "ransac_ms": t["ransac"],
                          "ransac_pct_kernel": share["ransac"],
                          "top_stage": max(share, key=share.get), "kernel_ms": t["kernel_total"],
                          "e2e_ms": t["end_to_end"]})
for row in BASELINE_ROWS:
    print(f"V5 {row['case']}: S5={row['ransac_ms']:.2f} ms ({row['ransac_pct_kernel']:.1f}% kernel), "
          f"top stage={row['top_stage']}, kernel={row['kernel_ms']:.2f} ms, E2E={row['e2e_ms']:.2f} ms")

## 5. Benchmark V0 / V5 / V7 trên đủ 12 cặp ảnh

Mỗi case chạy parity thật (`build/test_v7_pipeline <left> <right>`), sau đó đo V0, V5, V7 (đảo thứ tự V5/V7 xen kẽ theo scene, mỗi lần 1 warm-up + `RUNS` timed runs).

In [ ]:
SCENES = ["location1", "location2", "location3", "challenge"]
RESOLUTIONS = [("720p", "small_720p"), ("1080p", "medium_1080p"), ("4k", "large_4k")]

summary_rows = []
timing_rows = []

for scene_idx, scene in enumerate(SCENES):
    for res_label, folder in RESOLUTIONS:
        case = f"{scene}_{res_label}"
        left = Path("dataset") / scene / folder / "1.jpg"
        right = Path("dataset") / scene / folder / "2.jpg"
        print(f"\n=== CASE {case} ===", flush=True)
        if not left.is_file() or not right.is_file():
            summary_rows.append({"case": case, "status": "failed", "parity": "missing_dataset"})
            continue
        p_code = run(["build/test_v7_pipeline", left, right], check=False, log=RESULTS / f"{case}_parity.log")
        parity_status = "pass" if p_code == 0 else "failed"
        order = (0, 7, 5) if (scene_idx % 2) else (0, 5, 7)
        reports = {}
        ok = (p_code == 0)
        for v in order:
            r_path = RESULTS / f"{case}_v{v}.json"
            c_ret = run(["build/stitch_v7", left, right, RESULTS / f"{case}_v{v}.png",
                         "--version", v, "--runs", RUNS, "--json", r_path],
                        check=False, log=RESULTS / f"{case}_v{v}.log")
            if c_ret != 0 or not r_path.is_file():
                ok = False
                continue
            rep = json.loads(r_path.read_text(encoding="utf-8"))
            reports[v] = rep
            t = rep["timings_ms"]
            timing_rows.append({"case": case, "scene": scene, "resolution": res_label, "version": v,
                                "status": rep.get("status", "ok"), "parity": parity_status,
                                "inliers": rep["features"]["inliers"], **t})
        if len(reports) == 3:
            t0, t5, t7 = [reports[v]["timings_ms"] for v in (0, 5, 7)]
            same_inliers = reports[5]["features"]["inliers"] == reports[7]["features"]["inliers"] == reports[0]["features"]["inliers"]
            summary_rows.append({
                "case": case, "resolution": res_label, "status": "ok" if ok else "failed",
                "parity": parity_status, "inliers_equal_v0": same_inliers,
                "inliers": reports[7]["features"]["inliers"], "matches": reports[7]["features"]["matches"],
                "v0_s5_ms": t0["ransac"], "v5_s5_ms": t5["ransac"], "v7_s5_ms": t7["ransac"],
                "s5_speedup_v5_v7": t5["ransac"] / t7["ransac"] if t7["ransac"] > 0 else float("nan"),
                "v5_s5_pct_kernel": 100 * t5["ransac"] / t5["kernel_total"],
                "v7_s5_pct_kernel": 100 * t7["ransac"] / t7["kernel_total"],
                "v5_kernel_ms": t5["kernel_total"], "v7_kernel_ms": t7["kernel_total"],
                "v0_e2e_ms": t0["end_to_end"], "v5_e2e_ms": t5["end_to_end"], "v7_e2e_ms": t7["end_to_end"],
                "e2e_speedup_v5_v7": t5["end_to_end"] / t7["end_to_end"],
                "e2e_speedup_v0_v7": t0["end_to_end"] / t7["end_to_end"],
            })
        else:
            summary_rows.append({"case": case, "resolution": res_label, "status": "failed", "parity": parity_status})

import pandas as pd
summary = pd.DataFrame(summary_rows)
timings = pd.DataFrame(timing_rows)
summary.to_csv(RESULTS / "summary.csv", index=False)
timings.to_csv(RESULTS / "timings.csv", index=False)
print("Saved:", RESULTS / "summary.csv")

## 6. So sánh V5 vs V7: S5, kernel và end-to-end

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
ok_sum = summary.loc[summary.status.eq("ok") & summary.parity.eq("pass")].copy()
print(f"Verified cases: {len(ok_sum)}/{len(summary)}")

print("=== 6.1 S5 RANSAC (ms) và tỷ trọng trong kernel ===")
display(ok_sum[["case", "matches", "inliers", "inliers_equal_v0", "v0_s5_ms", "v5_s5_ms", "v7_s5_ms",
                "s5_speedup_v5_v7", "v5_s5_pct_kernel", "v7_s5_pct_kernel"]].round(3))

print("=== 6.2 Kernel và End-to-End ===")
display(ok_sum[["case", "v5_kernel_ms", "v7_kernel_ms", "v5_e2e_ms", "v7_e2e_ms",
                "e2e_speedup_v5_v7", "e2e_speedup_v0_v7"]].round(3))

print("=== 6.3 Geometric mean theo độ phân giải ===")
by_res = ok_sum.groupby("resolution", sort=False)[["s5_speedup_v5_v7", "e2e_speedup_v5_v7",
                                                    "e2e_speedup_v0_v7"]].agg(geometric_mean)
display(by_res.round(3))

for cols, ylabel in [
    (["v5_s5_ms", "v7_s5_ms"], "S5 RANSAC time (ms)"),
    (["v5_e2e_ms", "v7_e2e_ms"], "End-to-End time (ms)"),
]:
    ok_sum.set_index("case")[cols].plot.bar(figsize=(14, 4), ylabel=ylabel)
    plt.tight_layout()
    plt.show()

## 7. Nsight Systems: kernel S5 V5 vs V7

So `score_hypotheses_kernel` + `refine_best_kernel` (V5) với `estimate_hypotheses_v7_kernel` + `score_hypotheses_warp_v7_kernel` + `refine_best_warp_v7_kernel` (V7); `generate_samples_kernel`, `validate_matches_kernel`, `compact_points_kernel` giống nhau.

In [ ]:
PROFILE_DIR = SESSION_DIR / "profiles"
PROFILE_DIR.mkdir(exist_ok=True)
PROFILE_CASES = [("location1", "small_720p", "location1_720p"),
                 ("location1", "large_4k", "location1_4k")]
S5_KERNELS = ("validate_matches", "compact_points", "generate_samples", "score_hypotheses",
              "refine_best", "estimate_hypotheses")

def parse_nsys_csv(text, required_cols):
    rows, cols = [], None
    for vals in csv.reader(text.splitlines()):
        vals = [v.strip().lstrip("\ufeff") for v in vals]
        if required_cols <= set(vals):
            cols = vals
        elif cols is not None and len(vals) == len(cols):
            rows.append(dict(zip(cols, vals)))
    return rows

def profile_kernels(scene, folder, case, version):
    prefix = PROFILE_DIR / f"nsys_v{version}_{case}"
    left = Path("dataset") / scene / folder / "1.jpg"
    right = left.with_name("2.jpg")
    run(["nsys", "profile", "--trace=cuda", "--sample=none", "--force-overwrite=true",
         "-o", prefix, "build/stitch_v7", left, right,
         PROFILE_DIR / f"{case}_v{version}.png", "--version", version, "--runs", RUNS],
        log=prefix.with_suffix(".log"))
    log_f = prefix.with_name(f"{prefix.name}_cuda_gpu_kern_sum.log")
    run(["nsys", "stats", "--force-export=true", "--report", "cuda_gpu_kern_sum", "--format", "csv",
         "--output", "-", prefix.with_suffix(".nsys-rep")], log=log_f)
    df = pd.DataFrame(parse_nsys_csv(log_f.read_text(encoding="utf-8"), {"Total Time (ns)", "Instances", "Name"}))
    df.to_csv(prefix.with_name(f"{prefix.name}_cuda_gpu_kern_sum.csv"), index=False)
    df["total_ms_per_run"] = df["Total Time (ns)"].str.replace(",", "").astype(float) / 1e6 / (RUNS + 1)
    total = df["total_ms_per_run"].sum()
    s5 = df.loc[df["Name"].str.contains("|".join(S5_KERNELS), na=False)].copy()
    s5["pct_all_kernels"] = 100 * s5["total_ms_per_run"] / total
    print(f"--- V{version} {case}: S5 kernels (ms / pipeline run) ---")
    display(s5[["Name", "Instances", "total_ms_per_run", "pct_all_kernels"]].round(4))
    return {"case": case, "version": version, "s5_kernel_ms": s5["total_ms_per_run"].sum(),
            "all_kernel_ms": total, "s5_pct": 100 * s5["total_ms_per_run"].sum() / total}

if RUN_PROFILER and shutil.which("nsys"):
    prof_rows = []
    for scene, folder, case in PROFILE_CASES:
        for version in (5, 7):
            prof_rows.append(profile_kernels(scene, folder, case, version))
    prof_df = pd.DataFrame(prof_rows)
    prof_df.to_csv(PROFILE_DIR / "v5_vs_v7_nsys_s5_summary.csv", index=False)
    print("=== Tóm tắt Nsight S5 V5 vs V7 (trung bình mỗi pipeline run) ===")
    display(prof_df.round(3))
else:
    print("NOT RUN: nsys unavailable.")

## 8. Tổng kết V7

Kiểm tra lại:
1. Parity: V7 trùng V5 từng bit và đạt tiêu chí PLAN với V0 trên đủ 12 case (số inlier, H, panorama).
2. Thời gian S5 (`timings_ms.ransac`) và kernel S5 trong Nsight giảm từ V5 sang V7; tỷ trọng S5 trong kernel sau V7.
3. Speedup end-to-end V5→V7 và V0→V7 theo từng độ phân giải. Nếu S5 nhỏ so với tổng, speedup E2E nhỏ là đúng kỳ vọng (Amdahl); ghi lại kernel nào trở thành bottleneck mới.